<a href="https://colab.research.google.com/github/RakhaFS/Dataset-Polisi/blob/main/BERTopic_n_IndoBERTweet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Modul Import

In [ ]:
!pip install bertopic
!pip install sentence-transformers
!pip install Sastrawi
!pip install torch
!pip install transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.5 MB/s eta 0:00:00


##Data

In [ ]:
import pandas as pd
import re

#A = pd.read_csv("https://raw.githubusercontent.com/RakhaFS/Dataset-Polisi/refs/heads/main/Polisi.csv")
A = pd.read_csv("https://raw.githubusercontent.com/RakhaFS/Dataset-Polisi/refs/heads/main/tweet_pol_indo.csv")
df = A[['created_at','favorite_count','full_text']]
df

,created_at,favorite_count,full_text
0,Fri Jan 31 23:43:19 +0000 2020,0,Alhamdulillah sudah Tertangkap Semoga tidak ad...
1,Fri Jan 31 23:42:12 +0000 2020,0,@jejakshinta @_rstukrnwn @EletrikPrick @dwikap...
2,Fri Jan 31 23:42:02 +0000 2020,8,Polisi sebut kematian istri Sule Lina bukan ka...
3,Fri Jan 31 23:42:02 +0000 2020,15,Setelah 22 hari pemeriksaan laboratorium polis...
4,Fri Jan 31 23:41:20 +0000 2020,1,@Rang_Hoki @Offa_ @haibebyjo @jokowi Maksudnya...
...,...,...,...
64957,Wed Dec 31 08:05:55 +0000 2025,0,Kapolri Jenderal Polisi Drs. Listyo Sigit Prab...
64958,Wed Dec 31 08:05:54 +0000 2025,2,Kapolri Jenderal Polisi Drs. Listyo Sigit Prab...
64959,Wed Dec 31 08:05:54 +0000 2025,0,@arekgerak Lapor polisi bukan lapor medsos. La...
64960,Wed Dec 31 08:05:48 +0000 2025,0,terima kasih pak polisi sudah bekerja keras se...


##Preprocessing

In [ ]:
import requests

def load_slang_from_github(url):
    slang_dict = {}
    response = requests.get(url)
    for line in response.text.splitlines():
        if ":" in line:
            slang, formal = line.strip().split(":", 1)
            slang_dict[slang] = formal
    return slang_dict

slang_url = "https://raw.githubusercontent.com/RakhaFS/Dataset-Polisi/refs/heads/main/20210315_slangword.txt"
slang_dict = load_slang_from_github(slang_url)

def replace_slang(text, slang_dict):
    words = text.split()
    return " ".join([slang_dict.get(word, word) for word in words])


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+","",text)     # hapus URL
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)  # hapus emoji/simbol
    text = re.sub(r"\s+", " ", text).strip()
    text = replace_slang(text, slang_dict)
    text = re.sub(r"\d+", "", text)
    return text

df["clean_text"] = df["full_text"].apply(clean_text)
df.head()

/tmp/ipykernel_3056/431814824.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["clean_text"] = df["full_text"].apply(clean_text)


,created_at,favorite_count,full_text,clean_text
0,Fri Jan 31 23:43:19 +0000 2020,0,Alhamdulillah sudah Tertangkap Semoga tidak ad...,alhamdulillah sudah tertangkap semoga tidak ad...
1,Fri Jan 31 23:42:12 +0000 2020,0,@jejakshinta @_rstukrnwn @EletrikPrick @dwikap...,jejakshinta rstukrnwn eletrikprick dwikaputra ...
2,Fri Jan 31 23:42:02 +0000 2020,8,Polisi sebut kematian istri Sule Lina bukan ka...,polisi sebut kematian istri sule lina bukan ka...
3,Fri Jan 31 23:42:02 +0000 2020,15,Setelah 22 hari pemeriksaan laboratorium polis...,setelah hari pemeriksaan laboratorium polisi ...
4,Fri Jan 31 23:41:20 +0000 2020,1,@Rang_Hoki @Offa_ @haibebyjo @jokowi Maksudnya...,rang hoki offa haibebyjo jokowi maksudnya bisa...


In [ ]:
import requests

response = requests.get("https://raw.githubusercontent.com/RakhaFS/Dataset-Polisi/main/stopword_id.txt")

stopwords_id = set(response.text.splitlines())

query_words = [
    "polisi", "isilop", "polri",
    "aparat", "pakpol", "kapolri", "pid", "ep",
    "kepolisian", "indonesia", "tni", "kecuali", "haribhayangkarake",
    "drs", "democrazymedia", "pn", "tjandra", "dirgakkum", "mama", "celine",
    "sigit", "elsa", "listyo", "divhumas", "lina", "menurutmu", "haruka",
    "tirta", "rocky", "marquez", "na", "geloraco", "trenggalek", "agnes",
    "sule", "bojonegoro", "jessi", "alextham", "gerung", "cepot", "urs",
    "daftarkan", "susipudjiastuti", "prabowo", "spripim", "nikita", "polresnganjuk",
    "lampung", "fil", "santorinissun", "din", "edimulyadi", "bepergian", "mirzani",
    "humastabessmg", "dandelion", "megamendung", "ditelfon", "alex", "bisaaa", "tomat",
    "antir", "ully", "xixixixiixixix", "mulutmu", "fiony", "firli", "mart", "salam",
    "telpon", "humaspoldajatim", "habiebbass", "alexis", "presisiuntukindonesia",
    "nagregpolsek", "rhaters", "humaskedirires", "ikatancintaep", "polripresisi",
    "organik", "bahuri", "poldajogja", "name", "partner", "restulungagung",
    "tis", "dumdum", "haribhayangkaratahunmengabdi", "millen", "aiman", "bhayangkara",
    "ssdm", "polresbojonegoro", "bercyandaaa", "pling", "woyy", "jokowi", "listyosigitp",
    "divisihumaspolri", "cnnindonesia", "djoko", "fotonya", "helmy", "anak", "suporter",
    "dennysiregar", "ibunya", "edy", "cyrus", "santika", "mohmahfudmd", "sapi", "diraja",
    "siaranbolalive", "banget", "ccic", "blt", "nek", "denny", "rizieq", "pssi",
    "dewasasampaikanaspirasi", "islam", "tempodotco", "hutpolwanth", "kebhinekaan",
    "tvonenews", "najwashihab", "india", "israel", "sedini", "china", "malaysia",
    "tiongkok", "segi", "palestina", "habib", "rekrutmen", "dennyindrayana", "radioelshinta",
    "xi", "lesti", "kompastv", "heraloebss", "rekrutmenpolri", "txtdrberseragam",
    "detikcom", "ahok", "jubaedah", "terimakasih", "gaza", "anaknya", "dewahoya",
    "cina", "ringankan", "aremania", "gojekmilitan", "wni", "penerimaanpolri", "netralitaspolri",
    "billar", "jerman", "opsketupatkrakatau", "penerimaananggotapolri", "kejora",
    "humanistegakkanhukum", "yerusalem", "pemiluindonesiadamai", "gantengmaryana", "ccicpolri",
    "polisikendal", "semeru", "ngeprank", "poldajawatimur", "mira", "ayu", "sukoharjo", "zahra", "sean",
    "txtdrimedia", "thalia", "aqsa", "saban", "pulang", "zee", "tka", "shihab", "wn", "rusia",
    "polritangkapdjokotjandra", "hamas", "polres", "ara", "gegana", "ting", "rizky", "raya", "kedubes",
    "polsek", "ukraina", "november", "semangat", "itaewon", "bekerjasama", "diman", "sari", "operasililincandi",
    "even", "nataru", "kamboja", "papua", "bjn", "cakra", "sekedar", "persebaya", "pol", "lapan", "sus",
    "teddgus", "sing", "cirebonpolresta", "seng", "sekelik", "dadi", "pelayananprima", "siregar", "korea",
    "jogo", "pencak", "bahar", "iku", "grand", "turki", "yusuf", "opo", "arema", "ditelpon", "sek", "krakatau",
    "tahunbaru", "multimediahumaspoldaaceh", "sepuluh", "stakeholder", "ponpes", "myanmar", "kepala", "neng",
    "wae", "kartu", "ora", "prank", "halloween", "ono", "dewe", "saiki", "kamp"
    ]

custom_stopwords = set(query_words)
all_stopwords = stopwords_id.union(custom_stopwords)


In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    stop_words=list(all_stopwords),
    ngram_range=(1,1),
    min_df=2,
    max_df=0.9
)

In [ ]:
df

,created_at,favorite_count,full_text,clean_text,topic
0,Fri Jan 31 23:43:19 +0000 2020,0,Alhamdulillah sudah Tertangkap Semoga tidak ad...,alhamdulillah sudah tertangkap semoga tidak ad...,1
1,Fri Jan 31 23:42:12 +0000 2020,0,@jejakshinta @_rstukrnwn @EletrikPrick @dwikap...,jejakshinta rstukrnwn eletrikprick dwikaputra ...,-1
2,Fri Jan 31 23:42:02 +0000 2020,8,Polisi sebut kematian istri Sule Lina bukan ka...,polisi sebut kematian istri sule lina bukan ka...,23
3,Fri Jan 31 23:42:02 +0000 2020,15,Setelah 22 hari pemeriksaan laboratorium polis...,setelah hari pemeriksaan laboratorium polisi ...,23
4,Fri Jan 31 23:41:20 +0000 2020,1,@Rang_Hoki @Offa_ @haibebyjo @jokowi Maksudnya...,rang hoki offa haibebyjo jokowi maksudnya bisa...,2
...,...,...,...,...,...
64957,Wed Dec 31 08:05:55 +0000 2025,0,Kapolri Jenderal Polisi Drs. Listyo Sigit Prab...,kapolri jenderal polisi drs listyo sigit prabo...,2
64958,Wed Dec 31 08:05:54 +0000 2025,2,Kapolri Jenderal Polisi Drs. Listyo Sigit Prab...,kapolri jenderal polisi drs listyo sigit prabo...,-1
64959,Wed Dec 31 08:05:54 +0000 2025,0,@arekgerak Lapor polisi bukan lapor medsos. La...,arekgerak lapor polisi bukan lapor medsos lagi...,-1
64960,Wed Dec 31 08:05:48 +0000 2025,0,terima kasih pak polisi sudah bekerja keras se...,terima kasih pak polisi sudah bekerja keras se...,-1


##Embedding IndoBERTweet

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

docs = df["clean_text"].tolist()
model_embed = SentenceTransformer("firqaaa/indo-sentence-bert-base")
embeddings = model_embed.encode(docs, show_progress_bar=True)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/118 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2031 [00:00<?, ?it/s]

In [ ]:
df_final = df_akhir.merge(
    df_awal[["clean_text", "created_at"]],
    left_on="text",
    right_on="clean_text",
    how="left"
)

##BERTopic

In [ ]:
topic_model = BERTopic(
    nr_topics=25,
    top_n_words=1,          # 🔥 1 kata per topik
    min_topic_size=100,
    verbose=True,
    vectorizer_model=vectorizer,
    language="indonesian"
)

#topics, probs = topic_model.fit_transform(docs, embeddings)
topics, probs = topic_model.transform(docs)
df["topic"] = topics


ValueError: This BERTopic instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [ ]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,34112,-1_lapor,[lapor],[etherealglowy areajulid aku lebih setuju cara...
1,0,7181,0_uang,[uang],[andipitopang masuk polisi pakai uang kalii se...
2,1,6933,1_pelaku,[pelaku],[viral oknum polisi lampung curi mobil perwira...
3,2,6601,2_selamat,[selamat],[kepala kepolisian daerah jawa timur beserta s...
4,3,1756,3_presiden,[presiden],[sabar itu berani menunjukkan ijazah asli kok ...
5,4,1590,4_film,[film],[idioticstupidt kalau nonton semua drama polis...
6,5,1029,5_tidur,[tidur],"[rasanya jadi polisi tidur bagaimana ya, zefan..."
7,6,808,6_pembunuh,[pembunuh],[polisi pembunuh polisi pembunuh polisi pembun...
8,7,781,7_tanyarlfes,[tanyarlfes],"[roains bukti lapor kepada polisi tidak, tanya..."
9,8,642,8_lek,[lek],[dedeadestya wis radue foto gek cah cah do mle...


In [ ]:
  topic_model.get_topic(7)

##Mapping Topic

In [ ]:
aspect_dict = {
    "kinerja": [0, 3], #Liat dlu topicnya begimana kelompokkkinnya gimana
    "pungli": [1],
    "kekerasan": [2],
    "netralitas": [4]
}

def map_aspect(topic):
    for aspect, topic_list in aspect_dict.items():
        if topic in topic_list:
            return aspect
    return "lainnya"

df["aspect"] = df["topic"].apply(map_aspect)
df.head()


NameError: name 'df' is not defined

##Sentimen IndoBERTweet

In [ ]:
indobenchmark/indobertweet-sentiment

NameError: name 'indobenchmark' is not defined

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobertweet-sentiment")
model_sent = AutoModelForSequenceClassification.from_pretrained("indobenchmark/indobertweet-sentiment")

labels = ["negative", "neutral", "positive"]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


OSError: indobenchmark/indobertweet-sentiment is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model_sent(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    return labels[torch.argmax(probs)]

df["sentiment"] = df["clean_text"].apply(predict_sentiment)